> **Repository execution note.** Start JupyterLab from the **repository root** and place authorised source data in that same root directory (see [README](../README.md) and [Data requirements](../data/README.md)). These notebooks were originally distributed as standalone packages: any older instructions referring to an included ZIP, a `test/` folder, or bundled data do not apply to this GitHub repository. The notebooks preserve their original input schemas and experiment configurations. Results cached in the original files are intentionally excluded from this public-ready copy.

# Notebook 07 | Final Integrated Spatio-Temporal Benchmark
### Luís Silvano · Hotel Labour Productivity Project · *Knowledge-Based Systems*

**Scientific question:** *How do spatial accessibility, temporal memory and economic disruptions jointly contribute to the prediction and interpretation of hotel labour productivity?*

This notebook consolidates earlier studies into one protocol, **using the same sample across scenarios A–E**. The outcome is hotel productivity in year **t**, predicted from economic and structural characteristics available at **t−1** and time-invariant spatial indicators collected at a date still to be confirmed.

| Scenario | Information introduced | Hypothesis |
|:---|:---|:---|
| **A — Structure** | Hotel attributes, region, year | Baseline |
| **B — Structure + space** | A + accessibility and service indicators | H1: incremental value of accessibility |
| **C — Structure + memory** | A + productivity at t−1 | H2: value of temporal memory |
| **D — Space + memory** | C + continuous spatial indices | Informational interaction between space and time |
| **E — Space + memory + clusters** | D + clusters fitted **on training data only** | H3: profile-based representation |

**Important:** cluster membership is a function of spatial features and is not a new source of information. Scenario E tests whether that representation is beneficial to the model's learning algorithm. Model selection uses 2021–2023, but these years were explored previously. **2024 was also examined in earlier notebooks and is used solely for a retrospective audit.** This package has no completely blind test set.

**Historical limitation:** point-of-interest locations may reflect conditions in 2026 rather than 2018–2024. Without historical POI records, the results are retrospective comparisons, **not strictly contemporaneously feasible forecasts**. Association does not imply causation.


## 0. Execution guide

1. Extract the package into the working folder. The notebook uses the included `hoteis_2018_2024_subset.csv` and `indicadores_espaciais_05.csv`, **but prioritises** the updated Notebook 05 files inside `resultados_espaciais_05/`.
2. If you have run Notebook 05 with complete distance matrices for castles, churches, and beaches, copy the `resultados_espaciais_05` folder next to this notebook, especially `09_indicadores_e_clusters_por_hotel.csv` and `10_painel_com_indicadores_espaciais.csv`. The notebook automatically selects `k=5` when all three indices exist and have adequate coverage.
3. Run **Kernel → Restart Kernel and Run All**. When it finishes, open `resultados_KBS07` to find CSV tables, PNG figures, predictions, and execution reports.
4. Basic installation: `%pip install pandas numpy scipy scikit-learn matplotlib seaborn xgboost catboost openpyxl nbformat`.
5. For tabular deep learning (optional): `%pip install "pytabkit[models]"`. These models may be slow on a CPU; adjust `PERFIL` in the next cell.

**Profiles:** `verificacao` runs Ridge, Random Forest, and XGBoost; `principal` adds CatBoost (a relatively fast run); `completo` adds RealMLP and FT-Transformer (**the delivered file's default**). Failed or unavailable models are recorded and never silently omitted from the reports.


## 1. Fixed parameters and libraries

The following choices reflect a **retrospective confirmatory stage**, based on the preceding exploratory notebooks. New modifications should be documented as additional analyses, not selected because they improve 2024 results.


In [ ]:
from pathlib import Path
import os, sys, json, time, traceback, importlib.util, warnings, platform, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error, r2_score
from sklearn.dummy import DummyRegressor

ROOT = Path.cwd()
OUT = ROOT/'resultados_KBS07'
OUT.mkdir(parents=True, exist_ok=True)
SEED=42
THREADS=2
PERFIL=os.getenv('KBS07_PERFIL','completo')  # verificacao | principal | completo (quick check | main | full)
MODELOS={'verificacao':['ridge','random_forest','xgboost'],
         'principal':['ridge','random_forest','xgboost','catboost'],
         'completo':['ridge','random_forest','xgboost','catboost','realmlp','ft_transformer']}[PERFIL]
ANOS_CV=[2021, 2022, 2023]
ANO_AUDITORIA=2024
EXECUTAR_AUDITORIA_2024=True
N_BOOT=int(os.getenv('KBS07_BOOT','300'))
K_CLUSTERS=5
PERCENTIL_Y=0.995  # upper threshold computed exclusively from training y; never changes validation/test y
EXECUTAR_SENSIBILIDADE=True
EXECUTAR_SHAP = os.getenv('KBS07_SHAP','1')=='1'
CORES={'A':'#607D8B','B':'#3178B4','C':'#D58A25','D':'#119977','E':'#A66CBD'}
plt.rcParams.update({'figure.dpi':110,'savefig.dpi':200,'font.size':10,'axes.titlesize':12})
np.random.seed(SEED)
print('Configuration:',PERFIL, '| Models:',MODELOS, '| Audit year:',ANO_AUDITORIA)
print('Working directory:',ROOT.resolve())

## 2. Data provenance and automatic loading

The join key is the hotel's registration number (`ht_registration_number` ↔ `hotel_key`), **not** `ht_id`. The `10_painel_com_indicadores_espaciais.csv` table from Notebook 05 may already contain spatial indices and clusters; they are removed before merging to avoid duplicate columns. Clusters originally calculated in Notebook 05 are used **only for descriptive analyses**.


In [ ]:
def select_existing_file(caminhos,descricao):
    achado=next((p for p in caminhos if p.is_file()),None)
    if achado is None:
        raise FileNotFoundError(f'{descricao} was not found. Checked paths: '+ ', '.join(str(p) for p in caminhos))
    return achado

def normalize_hotel_id(x):
    if pd.isna(x): return None
    try:
        f=float(x)
        if np.isfinite(f) and f.is_integer(): return str(int(f))
    except (ValueError,TypeError): pass
    return str(x).strip() or None

# Accept Notebook 07 in test/ OR in the test/KBS07_Benchmark_Integrado/ subfolder.
# Prefer complete Notebook 05 results, even when located in the parent folder.
ARQ_PAINEL=select_existing_file([
    ROOT/'resultados_espaciais_05'/'10_painel_com_indicadores_espaciais.csv',
    ROOT.parent/'resultados_espaciais_05'/'10_painel_com_indicadores_espaciais.csv',
    ROOT/'10_painel_com_indicadores_espaciais.csv',
    ROOT.parent/'10_painel_com_indicadores_espaciais.csv',
    ROOT/'hoteis_2018_2024_subset.csv',
    ROOT/'KBS07_Benchmark_Integrado'/'hoteis_2018_2024_subset.csv',
    ROOT.parent/'hoteis_2018_2024_subset.csv',
], 'Annual panel')
ARQ_ESPACO=select_existing_file([
    ROOT/'resultados_espaciais_05'/'09_indicadores_e_clusters_por_hotel.csv',
    ROOT.parent/'resultados_espaciais_05'/'09_indicadores_e_clusters_por_hotel.csv',
    ROOT/'09_indicadores_e_clusters_por_hotel.csv',
    ROOT.parent/'09_indicadores_e_clusters_por_hotel.csv',
    ROOT/'indicadores_espaciais_05.csv',
    ROOT/'KBS07_Benchmark_Integrado'/'indicadores_espaciais_05.csv',
    ROOT.parent/'indicadores_espaciais_05.csv',
], 'Indicators from Notebook 05')

painel=pd.read_csv(ARQ_PAINEL,low_memory=False)
espaco=pd.read_csv(ARQ_ESPACO,low_memory=False)
NECESSARIAS=['f_year','f_labprodct','ht_hotel_chain_d','ht_capacity','ht_quality_stars','ht_nuts_2_newcriteria']
if any(c not in painel for c in NECESSARIAS):
    raise ValueError('Missing annual columns: '+str(set(NECESSARIAS)-set(painel.columns)))
if 'hotel_key' not in espaco.columns:
    raise ValueError('Spatial indicators lack hotel_key')
if 'hotel_key' not in painel.columns:
    if 'ht_registration_number' not in painel:
        raise ValueError('Panel lacks hotel_key and ht_registration_number')
    painel['hotel_key']=painel.ht_registration_number.map(normalize_hotel_id)
else:
    painel['hotel_key']=painel.hotel_key.map(normalize_hotel_id)
espaco['hotel_key']=espaco.hotel_key.map(normalize_hotel_id)
if espaco.hotel_key.isna().any() or espaco.hotel_key.duplicated().any():
    raise ValueError('Missing or duplicated identifier in spatial file')

K5=[f'a_{c}_k5' for c in ['castelos','igrejas','praias']]
K1=[f'a_{c}_k1' for c in ['castelos','igrejas','praias']]
K5_DISPONIVEL=all(c in espaco.columns and pd.to_numeric(espaco[c],errors='coerce').notna().mean()>=.95 for c in K5)
K_USADO=5 if K5_DISPONIVEL else 1
SP_COLS=[f'a_{c}_k{K_USADO}' for c in ['castelos','igrejas','praias']]+['log_dens_restaurantes_r0p8km']
if any(c not in espaco for c in SP_COLS):
    raise ValueError('Missing spatial indicators: '+str(set(SP_COLS)-set(espaco.columns)))
for c in SP_COLS: espaco[c]=pd.to_numeric(espaco[c],errors='coerce')

# The main Notebook 05 file may contain older spatial columns (k=1).
# Drop all spatially derived columns and remerge with the current file 09.
DERIVADAS=[c for c in painel.columns if c.startswith(('a_castelos_k','a_igrejas_k','a_praias_k','log_dens_restaurantes_','dens_restaurantes_','n_restaurantes_r'))]
painel=painel.drop(columns=DERIVADAS+['cluster','perfil_nome'],errors='ignore')
for c in ['f_year','f_labprodct','ht_hotel_chain_d']:
    painel[c]=pd.to_numeric(painel[c],errors='coerce')
painel=painel.loc[painel.f_year.between(2018,2024)&painel.ht_hotel_chain_d.eq(0)&painel.hotel_key.notna()].copy()
painel['f_year']=painel.f_year.astype(int)
if painel.duplicated(['hotel_key','f_year']).any():
    raise ValueError('Panel contains duplicate hotel–year pairs')
base=painel.merge(espaco[['hotel_key']+SP_COLS],on='hotel_key',how='inner',validate='many_to_one')
base=base.replace([np.inf,-np.inf],np.nan)
base=base.loc[base[SP_COLS].notna().sum(axis=1)>=3].copy()
print('Panel:',ARQ_PAINEL.name,'|',len(painel),'observations')
print('Spatial data:',ARQ_ESPACO.name,'|',len(espaco),'hotels')
print('Number of neighbours:',K_USADO,'(k=5 only if complete for all three categories)')
print('Merge:',len(base),'observations |',base.hotel_key.nunique(),'hotels')
print('k=5 coverage:',{c:round(pd.to_numeric(espaco[c],errors='coerce').notna().mean(),3) if c in espaco else 0 for c in K5})

AUDIT_FONTE=pd.DataFrame({'fonte':['painel','indicadores','painel integrado'],
 'caminho':[str(ARQ_PAINEL),str(ARQ_ESPACO),'join'],
 'linhas':[len(painel),len(espaco),len(base)],
 'hoteis':[painel.hotel_key.nunique(),espaco.hotel_key.nunique(),base.hotel_key.nunique()]})
AUDIT_FONTE.to_csv(OUT/'01_proveniencia_dados.csv',index=False)
display(AUDIT_FONTE)

## 3. Constructing hotel–year pairs for one-step-ahead forecasting

For each target row in year `t`, hotel features and previous productivity are retrieved from **the same hotel's row in year `t-1`**. Both years must be present and have observed productivity. The final sample is identical across A, B, C, D, and E. **We do not use revenue, employee counts, or other financial components from `t` as predictors**, as they could encode the very definition of productivity.


In [ ]:
ESTRUTURA=['ht_capacity','ht_quality_stars','ht_number_of_accommodation_units',
           'ht_spa_d','ht_gym_d','ht_outdoor_pools_d','ht_meeting_rooms_d']
for c in ESTRUTURA:
    if c not in base: base[c]=np.nan
    base[c]=pd.to_numeric(base[c],errors='coerce')

prev=base[['hotel_key','f_year','f_labprodct','ht_nuts_2_newcriteria']+ESTRUTURA].copy()
prev=prev.rename(columns={'f_labprodct':'lag_y','ht_nuts_2_newcriteria':'regiao_lag',
                          **{c:'lag_'+c for c in ESTRUTURA}})
prev['f_year']=prev.f_year+1
atual=base[['hotel_key','f_year','f_labprodct']+SP_COLS].copy()
PARES=atual.merge(prev,on=['hotel_key','f_year'],how='inner',validate='one_to_one')
PARES=PARES.rename(columns={'f_labprodct':'y'}).replace([np.inf,-np.inf],np.nan)
PARES=PARES.dropna(subset=['y','lag_y']).copy()
PARES=PARES.loc[PARES.f_year.between(2019,2024)].sort_values(['f_year','hotel_key']).reset_index(drop=True)
PARES['regiao_lag']=PARES['regiao_lag'].fillna('Region not specified').astype(str)
PARES['ano_previsto']=PARES.f_year.astype(float)
PARES['ano_lag']=PARES.f_year-1
assert not PARES.duplicated(['hotel_key','f_year']).any()
assert PARES['ano_lag'].add(1).eq(PARES.f_year).all()
assert PARES['y'].notna().all() and PARES['lag_y'].notna().all()
assert PARES.hotel_key.isin(espaco.hotel_key).all()

AUDIT_PARES=PARES.groupby('f_year').agg(observacoes=('y','size'),hoteis=('hotel_key','nunique'),
                                        mediana_y=('y','median'),media_y=('y','mean')).reset_index()
AUDIT_PARES.to_csv(OUT/'02_auditoria_amostra_comum.csv',index=False)
print('Valid pairs:',len(PARES),'| hotels:',PARES.hotel_key.nunique())
display(AUDIT_PARES.round(2))

## 4. Mathematical formulation and methodological risks

For hotel \(i\), tourism category \(c\), and the \(k\) closest points of interest:

$
A_i^{(c)}(k)=\frac{1}{k}\sum_{j=1}^k\frac{1}{1+d_{ij}^{(c)}/d_0},\qquad d_0=1\,\mathrm{km}.
$

The restaurant/café count within an 800-metre radius is \(N_i(0.8)\); the approximate circular density and transformation are:

$
D_i(0.8)=\frac{N_i(0.8)}{\pi(0.8)^2}, \qquad R_i=\log(1+D_i(0.8)).
$

A general dynamic regression with exogenous information is:

$
\widehat{y}_{i,t}=f_\theta\left(\underbrace{\mathbf{s}_{i,t-1},r_{i,t-1},t}_{\mathrm{A}},
\underbrace{\mathbf{a}_i}_{\mathrm{B}},\underbrace{y_{i,t-1}}_{\mathrm{C}},
\underbrace{g_\theta(\mathbf{a}_i)}_{\mathrm{E}}\right).
$

In E, \(g_\theta\) is K-Means fitted exclusively to the hotels in each evaluation fold's training set. The cluster is one-hot encoded and concatenated with continuous variables. Every model uses the same sample, and transformations **are fitted on training data only**.

**Limitations:** POIs may be a *snapshot* taken after the study period; the historical panel does not support a strictly prospective spatial evaluation. Neither coefficients nor SHAP values represent causal effects. With few years, hotel-level bootstrap intervals do not eliminate the dependence arising from shocks shared by all hotels.


## 5. Tourism profiles and productivity trends (description, not training)

Clusters produced in Notebook 05 are used **only** for visualisation and identification of subpopulations. The clusters used in E are re-estimated within each fold, without consulting future data. Comparing medians over time is descriptive; the number of observed hotels may vary by year.


In [ ]:
if 'cluster' in espaco.columns:
    descritivo=base[['hotel_key','f_year','f_labprodct']].merge(
       espaco[['hotel_key','cluster']].dropna(subset=['cluster']),on='hotel_key',how='left',validate='many_to_one')
    descritivo['cluster']=pd.to_numeric(descritivo.cluster,errors='coerce')
    RESUMO_PERFIS=(descritivo.groupby(['cluster','f_year'],dropna=True)
                   .agg(n=('f_labprodct','count'),mediana=('f_labprodct','median'),
                        media=('f_labprodct','mean')).reset_index())
    RESUMO_PERFIS.to_csv(OUT/'03_evolucao_ano_por_perfil_descritivo.csv',index=False)
    fig,axs=plt.subplots(1,2,figsize=(13,4.5))
    for cl,g in RESUMO_PERFIS.groupby('cluster'):
        axs[0].plot(g.f_year,g.mediana,marker='o',label=f'Perfil {int(cl)}')
        axs[1].plot(g.f_year,g.n,marker='o',label=f'Perfil {int(cl)}')
    for ax in axs: ax.axvspan(2019.6,2020.4,color='#d0d0d0',alpha=.30);ax.set_xlabel('Year');ax.grid(alpha=.15)
    axs[0].set(ylabel='Median productivity',title='Productivity by spatial profile')
    axs[1].set(ylabel='Observations with productivity',title='Cobertura amostral')
    axs[0].legend(ncol=3,fontsize=8)
    plt.tight_layout(); plt.savefig(OUT/'figura_01_produtividade_por_perfil.png',bbox_inches='tight');plt.show()
    # Compare the same hotels before and during the pandemic shock.
    coorte=descritivo.loc[descritivo.f_year.isin([2019,2020])].pivot_table(
        index=['hotel_key','cluster'],columns='f_year',values='f_labprodct',aggfunc='first').dropna()
    coorte['variacao_2020_menos_2019']=coorte[2020]-coorte[2019]
    RESUMO_CHOQUE=(coorte.reset_index().groupby('cluster')['variacao_2020_menos_2019']
       .agg(n='count',mediana='median',media='mean',fracao_quedas=lambda x:(x<0).mean()).reset_index())
    RESUMO_CHOQUE.to_csv(OUT/'04_choque_2019_2020_por_perfil.csv',index=False)
    print('Paired 2019–2020 comparison within each profile:');display(RESUMO_CHOQUE.round(3))
else:
    descritivo=pd.DataFrame(); RESUMO_PERFIS=pd.DataFrame()
    print('Descriptive clusters from Notebook 05 are unavailable: profile plots cannot be generated.')

## 6. Training-only preprocessing and clusters without future information

Across A–E, structural predictors from year \(t-1\) are identical. Spatial indices are constant within a hotel and may have been collected in 2026. Robust treatment **clips only the training target at the 99.5th percentile** (without removing hotels). The observed validation target is left untouched. The `lag_y` predictor is itself clipped using percentiles calculated **from training data only**, reducing the influence of extremes without removing the temporal variable.


In [ ]:
BASE_NUM=['lag_'+x for x in ESTRUTURA]+['ano_previsto']
LAG_NUM=['lag_y']
CENARIOS={
 'A':BASE_NUM,
 'B':BASE_NUM+SP_COLS,
 'C':BASE_NUM+LAG_NUM,
 'D':BASE_NUM+LAG_NUM+SP_COLS,
 'E':BASE_NUM+LAG_NUM+SP_COLS,
}

class SpatialClusterFold:
    def __init__(self,k=5): self.k=k
    def fit(self,train):
        h=train.drop_duplicates('hotel_key')
        a=h[SP_COLS].astype(float).replace([np.inf,-np.inf],np.nan)
        self.imputer=SimpleImputer(strategy='median',keep_empty_features=True).fit(a)
        arr=self.imputer.transform(a)
        self.lo=np.quantile(arr,.01,axis=0); self.hi=np.quantile(arr,.99,axis=0)
        arr=np.clip(arr,self.lo,self.hi)
        self.scaler=StandardScaler().fit(arr)
        self.model=KMeans(n_clusters=self.k,random_state=SEED,n_init=12).fit(self.scaler.transform(arr))
        return self
    def predict(self,frame):
        arr=self.imputer.transform(frame[SP_COLS].astype(float).replace([np.inf,-np.inf],np.nan))
        arr=np.clip(arr,self.lo,self.hi)
        return self.model.predict(self.scaler.transform(arr))

class FeatureFold:
    def __init__(self,train,scenario,cluster_fit=None):
        self.scenario=scenario; self.cols=CENARIOS[scenario]; self.cluster_fit=cluster_fit
        self.imp=SimpleImputer(strategy='median',keep_empty_features=True)
        numeric=train[self.cols].astype(float).replace([np.inf,-np.inf],np.nan).copy()
        if 'lag_y' in numeric:
            self.lag_max=float(np.quantile(train.lag_y,.995))
            self.lag_min=float(np.quantile(train.lag_y,.005))
            numeric['lag_y']=numeric['lag_y'].clip(self.lag_min,self.lag_max)
        else: self.lag_min=self.lag_max=None
        a=self.imp.fit_transform(numeric)
        self.scale=StandardScaler().fit(a)
        try: self.ohe=OneHotEncoder(handle_unknown='ignore',sparse_output=False)
        except TypeError: self.ohe=OneHotEncoder(handle_unknown='ignore',sparse=False)
        self.ohe.fit(train[['regiao_lag']].astype(str))
        self.colnames=list(self.cols)+list(self.ohe.get_feature_names_out(['regiao_lag']))
        if scenario=='E':
            assert cluster_fit is not None
            self.colnames += [f'perfil_treino_{j}' for j in range(cluster_fit.k)]
    def transform(self,frame,scaled=False):
        z=frame[self.cols].astype(float).replace([np.inf,-np.inf],np.nan).copy()
        if 'lag_y' in z: z['lag_y']=z['lag_y'].clip(self.lag_min,self.lag_max)
        x=self.imp.transform(z)
        if scaled: x=self.scale.transform(x)
        reg=self.ohe.transform(frame[['regiao_lag']].astype(str))
        parts=[x,reg]
        if self.scenario=='E':
            cl=self.cluster_fit.predict(frame)
            parts.append(np.eye(self.cluster_fit.k,dtype=float)[cl])
        return np.hstack(parts).astype('float32')

# Programmatically verify that validation data do not enter transformer fitting.
def get_temporal_fold(ano,cenarios=CENARIOS):
    tr=PARES[PARES.f_year<ano].copy()
    va=PARES[PARES.f_year==ano].copy()
    if len(tr)<150 or len(va)<60: raise ValueError(f'Insufficient sample size to predict {ano}')
    assert tr.f_year.max()<ano and va.f_year.eq(ano).all()
    cluster_fit=SpatialClusterFold(K_CLUSTERS).fit(tr)
    data={}
    for c in cenarios:
        p=FeatureFold(tr,c,cluster_fit)
        data[c]={'tr_raw':p.transform(tr,False),'va_raw':p.transform(va,False),
                 'tr_scale':p.transform(tr,True),'va_scale':p.transform(va,True),
                 'colnames':p.colnames}
    return tr,va,data

def build_model(nome):
    if nome=='ridge': return Ridge(alpha=10.0)
    if nome=='random_forest': return RandomForestRegressor(n_estimators=180,min_samples_leaf=3,
                 max_features=.9,random_state=SEED,n_jobs=THREADS)
    if nome=='xgboost':
        from xgboost import XGBRegressor
        return XGBRegressor(n_estimators=280,max_depth=4,learning_rate=.045,
                 subsample=.9,colsample_bytree=.9,objective='reg:squarederror',
                 tree_method='hist',n_jobs=THREADS,random_state=SEED)
    if nome=='catboost':
        from catboost import CatBoostRegressor
        return CatBoostRegressor(iterations=350,depth=5,learning_rate=.045,
                 loss_function='RMSE',thread_count=THREADS,verbose=False,random_seed=SEED,
                 allow_writing_files=False)
    if nome in ['realmlp','ft_transformer']:
        import pytabkit
        cl={'realmlp':'RealMLP_TD_Regressor','ft_transformer':'FTT_D_Regressor'}[nome]
        return getattr(pytabkit,cl)(device='cuda' if importlib.util.find_spec('torch') and __import__('torch').cuda.is_available() else 'cpu',random_state=SEED)
    raise ValueError('Unknown model '+nome)

def estimate_predictions(nome,train,eval,mat,usar_clip=True):
    escalado=nome in ('ridge','realmlp','ft_transformer')
    xtr=mat['tr_scale'] if escalado else mat['tr_raw']
    xva=mat['va_scale'] if escalado else mat['va_raw']
    ytr=train.y.to_numpy(dtype=float).copy()
    teto=float(np.quantile(ytr,PERCENTIL_Y))
    if usar_clip: ytr=np.minimum(ytr,teto)
    model=build_model(nome)
    tic=time.perf_counter(); model.fit(xtr,ytr)
    pred=np.asarray(model.predict(xva)).reshape(-1)
    return pred,{'segundos':time.perf_counter()-tic,'teto_y_treino':teto}

def metrics(y,p):
    return {'MAE':mean_absolute_error(y,p),'RMSE':np.sqrt(mean_squared_error(y,p)),
            'R2':r2_score(y,p),'MedAE':median_absolute_error(y,p),
            'P90_AE':float(np.quantile(np.abs(np.asarray(y)-p),.90))}

print('Five scenarios defined:',{k:len(v) for k,v in CENARIOS.items()})

## 7. Walk-forward temporal validation — 2021, 2022, and 2023

Each validation year uses **only earlier target observations** for training. In each fold, all models use **the same validation sample**, with identical rows across all five scenarios. Persistence \(\hat y_{i,t}=y_{i,t-1}\) and the training mean serve as additional baselines. Beyond MAE, RMSE, and R², we save MedianAE and the 90th percentile of absolute error.


In [ ]:
CV=[]; PREDS=[]; LOG_FALHAS=[]
for ano in ANOS_CV:
    tr,va,prepare_feature_matrices=get_temporal_fold(ano)
    mean_train=float(tr.y.mean())
    baselines=[('media_treino','Z',np.repeat(mean_train,len(va))),
               ('persistencia','P',va.lag_y.to_numpy(float))]
    rmse_dummy=np.sqrt(mean_squared_error(va.y,np.repeat(mean_train,len(va))))
    print(f'\nVALIDATION {ano}: training={len(tr)} | validation={len(va)} | mean-baseline RMSE={rmse_dummy:.2f}')
    for nome,cenario,p in baselines:
        CV.append({'ano':ano,'modelo':nome,'cenario':cenario,'n_treino':len(tr),'n_validacao':len(va),
                   **metrics(va.y,p),'RMSE_rel_dummy':np.sqrt(mean_squared_error(va.y,p))/rmse_dummy,
                   'segundos':0.0})
        PREDS.append(pd.DataFrame({'hotel_key':va.hotel_key.to_numpy(),'ano':ano,'modelo':nome,
                                   'cenario':cenario,'real':va.y.to_numpy(),'predito':p}))
    for nome in MODELOS:
        for c,mat in prepare_feature_matrices.items():
            try:
                pred,info=estimate_predictions(nome,tr,va,mat)
                if not np.isfinite(pred).all(): raise ValueError('Predictions contain NaN/Inf')
                score=metrics(va.y,pred)
                CV.append({'ano':ano,'modelo':nome,'cenario':c,'n_treino':len(tr),'n_validacao':len(va),
                           **score,'RMSE_rel_dummy':score['RMSE']/rmse_dummy,**info})
                PREDS.append(pd.DataFrame({'hotel_key':va.hotel_key.to_numpy(),'ano':ano,
                    'modelo':nome,'cenario':c,'real':va.y.to_numpy(),'predito':pred}))
            except Exception as e:
                LOG_FALHAS.append({'ano':ano,'modelo':nome,'cenario':c,'etapa':'CV',
                    'tipo_erro':type(e).__name__,'erro':str(e)[:500],
                    'traceback':traceback.format_exc(limit=2)[:700]})
                print('  NOT EXECUTED:',nome,c,str(e)[:145])
    print('Models/scenarios executed for this year:',sum((r['ano']==ano) for r in CV))

CV_DF=pd.DataFrame(CV)
PREDS_DF=pd.concat(PREDS,ignore_index=True)
CV_DF.to_csv(OUT/'05_cv_temporal_modelos_cenarios.csv',index=False)
PREDS_DF.to_csv(OUT/'06_previsoes_cv_pareadas.csv',index=False)
pd.DataFrame(LOG_FALHAS,columns=['ano','modelo','cenario','etapa','tipo_erro','erro','traceback']).to_csv(OUT/'07_modelos_nao_executados.csv',index=False)
print('Total completed evaluations:',len(CV_DF),'| Individual predictions:',len(PREDS_DF))
display(CV_DF[CV_DF.cenario.isin(CENARIOS)].pivot_table(index=['modelo','cenario'],columns='ano',values='RMSE').round(2))

## 8. Validation-based ranking with fixed selection rules and incremental gains

**Do not use 2024 to select models.** A model must have results for all three validation years to enter the ranking. Before averaging across years, its RMSE in each year is divided by the RMSE of the training-mean baseline for that same year. This prevents the year with the largest absolute errors from dominating the temporal comparison. Paired analysis measures \(\Delta\mathrm{MAE}=\mathrm{MAE}_{\mathrm{new}}-\mathrm{MAE}_{\mathrm{previous}}\); negative values indicate improvement.


In [ ]:
RANKING=(CV_DF[CV_DF.cenario.isin(CENARIOS)].groupby(['modelo','cenario'],as_index=False)
         .agg(folds=('ano','nunique'),relativo=('RMSE_rel_dummy','mean'),
              MAE=('MAE','mean'),RMSE=('RMSE','mean'),R2=('R2','mean'),
              segundos=('segundos','sum')))
RANKING=RANKING[RANKING.folds.eq(len(ANOS_CV))].sort_values(['relativo','MAE']).reset_index(drop=True)
RANKING.to_csv(OUT/'08_ranking_modelos_e_configuracoes.csv',index=False)
if RANKING.empty: raise RuntimeError('No model completed all three validation years')
print('Full temporal-validation ranking (2021–2023):')
display(RANKING.head(16).round(3))

COMPARACOES=[('A','B','H1: space without memory'),('A','C','H2: memory without space'),
             ('C','D','H1: space with memory'),('B','D','H2: memory with space'),
             ('D','E','H3: cluster-based representation')]
GANHOS=[]; INDIVIDUAIS=[]
for nome in MODELOS:
    for a,b,h in COMPARACOES:
        l=PREDS_DF[(PREDS_DF.modelo==nome)&(PREDS_DF.cenario==a)]
        r=PREDS_DF[(PREDS_DF.modelo==nome)&(PREDS_DF.cenario==b)]
        both=l.merge(r,on=['hotel_key','ano'],suffixes=('_a','_b'),validate='one_to_one')
        if both.empty: continue
        assert np.allclose(both.real_a,both.real_b)
        both['delta_absoluto']=abs(both.predito_b-both.real_b)-abs(both.predito_a-both.real_a)
        both['modelo']=nome;both['comparacao']=f'{b}-{a}';both['hipotese']=h
        INDIVIDUAIS.append(both[['hotel_key','ano','modelo','comparacao','delta_absoluto']])
        for ano,g in both.groupby('ano'):
            GANHOS.append({'modelo':nome,'comparacao':f'{b}-{a}','hipotese':h,'ano':ano,
                           'n':len(g),'delta_MAE':g.delta_absoluto.mean(),
                           'fracao_b_melhor':(g.delta_absoluto<0).mean()})
GANHOS_DF=pd.DataFrame(GANHOS)
GANHOS_DF.to_csv(OUT/'09_ganhos_pareados_por_ano.csv',index=False)
DELTA_DF=pd.concat(INDIVIDUAIS,ignore_index=True) if INDIVIDUAIS else pd.DataFrame()
DELTA_DF.to_csv(OUT/'10_diferencas_absolutas_individuais.csv',index=False)
print('Gains by year (delta MAE; negative is better):')
display(GANHOS_DF[GANHOS_DF.modelo.isin(['random_forest','xgboost'])]
        .pivot_table(index=['modelo','comparacao'],columns='ano',values='delta_MAE').round(2))

## 9. Uncertainty in improvements: paired hotel-level bootstrap

We resample hotel identifiers (not isolated rows), retaining each selected hotel's observations across years. The interval quantifies variation between hotels **conditional on the three years studied**. It does not support confident inference about different macroeconomic shocks, because only three validation years are available.


In [ ]:
BOOT=[]
if not DELTA_DF.empty:
    rng=np.random.default_rng(SEED)
    for (modelo,comparacao),g in DELTA_DF.groupby(['modelo','comparacao']):
        grupos=[v.delta_absoluto.to_numpy(float) for _,v in g.groupby('hotel_key')]
        if len(grupos)<20: continue
        soma=np.array([x.sum() for x in grupos]); n=np.array([len(x) for x in grupos])
        estimativas=[]
        for _ in range(N_BOOT):
            idx=rng.integers(0,len(grupos),size=len(grupos))
            estimativas.append(soma[idx].sum()/n[idx].sum())
        BOOT.append({'modelo':modelo,'comparacao':comparacao,'delta_MAE':g.delta_absoluto.mean(),
                     'IC95_inf':np.quantile(estimativas,.025),'IC95_sup':np.quantile(estimativas,.975),
                     'hoteis':len(grupos),'n_pares':len(g)})
BOOT_DF=pd.DataFrame(BOOT).sort_values(['modelo','comparacao'])
BOOT_DF.to_csv(OUT/'11_bootstrap_hotel_delta_MAE.csv',index=False)
display(BOOT_DF.round(3))

## 10. Figures for the paper: performance and value of memory/space

The first figure presents predictive performance by year and scenario; the second displays marginal improvement (\(\Delta\)MAE) by hypothesis. We do not mix 2024 with the validation results.


In [ ]:
PRINCIPAIS=[m for m in ['xgboost','random_forest','catboost','ridge'] if m in CV_DF.modelo.unique()]
fig,axs=plt.subplots(1,2,figsize=(13,4.6))
sub=CV_DF[CV_DF.modelo.isin(PRINCIPAIS[:3])&CV_DF.cenario.isin(CENARIOS)]
sns.lineplot(data=sub,x='ano',y='RMSE_rel_dummy',hue='cenario',style='modelo',
             markers=True,dashes=False,palette=CORES,ax=axs[0])
axs[0].axhline(1,ls='--',c='gray',lw=.9);axs[0].set(title='Relative RMSE by year — same hotels within each fold',xlabel='Validation year',ylabel='RMSE / training-mean baseline RMSE')
axs[0].set_xticks(ANOS_CV)
delta_plot=GANHOS_DF[GANHOS_DF.modelo.isin(PRINCIPAIS[:3])&GANHOS_DF.comparacao.isin(['B-A','D-C','E-D'])]
sns.barplot(data=delta_plot,x='modelo',y='delta_MAE',hue='comparacao',errorbar=None,ax=axs[1])
axs[1].axhline(0,c='black',lw=.7);axs[1].set(title='Incremental gain by information block',xlabel='Model',ylabel='ΔMAE (negative = improvement)')
axs[1].tick_params(axis='x',rotation=10)
plt.tight_layout();plt.savefig(OUT/'figura_02_resultados_temporais_AB_CDE.png',bbox_inches='tight');plt.show()

fig,axs=plt.subplots(1,2,figsize=(13,4.4))
d=sns.barplot(data=GANHOS_DF[(GANHOS_DF.modelo=='xgboost') & (GANHOS_DF.comparacao.isin(['B-A','C-A','D-C','E-D']))],
             x='ano',y='delta_MAE',hue='comparacao',ax=axs[0],errorbar=None)
axs[0].axhline(0,color='black',lw=.7);axs[0].set(title='H1–H3 with XGBoost, by year',ylabel='ΔMAE',xlabel='Year')
cv_sub=CV_DF[(CV_DF.modelo=='xgboost')&CV_DF.cenario.isin(CENARIOS)]
sns.lineplot(data=cv_sub,x='ano',y='MAE',hue='cenario',palette=CORES,marker='o',ax=axs[1])
axs[1].set(title='MAE by scenario, XGBoost',xlabel='Year',ylabel='MAE');axs[1].set_xticks(ANOS_CV)
plt.tight_layout();plt.savefig(OUT/'figura_03_hipoteses_ano.png',bbox_inches='tight');plt.show()

## 11. Retrospective 2024 audit — **no new selection on the test set**

We evaluate only **the three highest-ranked combinations that completed validation over 2021–2023**, together with two predefined baselines. **2024 is not a blind test** in this project, because it has already influenced exploratory decisions. Claiming confirmatory results requires a later period or an independent external cohort.


In [ ]:
AUDIT24=[];PREDS24=[]
if EXECUTAR_AUDITORIA_2024:
    tr,va,mats=get_temporal_fold(ANO_AUDITORIA)
    dummy=np.repeat(tr.y.mean(),len(va)); persist=va.lag_y.to_numpy(float)
    for name,c,p in [('media_treino','Z',dummy),('persistencia','P',persist)]:
        AUDIT24.append({'modelo':name,'cenario':c,'ano':2024,'n':len(va),**metrics(va.y,p)})
        PREDS24.append(pd.DataFrame({'hotel_key':va.hotel_key.to_numpy(),'ano':2024,
           'modelo':name,'cenario':c,'real':va.y.to_numpy(),'predito':p}))
    SELECIONADOS=RANKING.head(3)[['modelo','cenario']].itertuples(index=False,name=None)
    for name,c in SELECIONADOS:
        try:
            p,extra=estimate_predictions(name,tr,va,mats[c])
            AUDIT24.append({'modelo':name,'cenario':c,'ano':2024,'n':len(va),**metrics(va.y,p),**extra})
            PREDS24.append(pd.DataFrame({'hotel_key':va.hotel_key.to_numpy(),'ano':2024,
                 'modelo':name,'cenario':c,'real':va.y.to_numpy(),'predito':p}))
        except Exception as e:
            LOG_FALHAS.append({'ano':2024,'modelo':name,'cenario':c,'etapa':'AUDITORIA',
                'tipo_erro':type(e).__name__,'erro':str(e)[:500]})
    AUDIT24_DF=pd.DataFrame(AUDIT24).sort_values('RMSE')
    PREDS24_DF=pd.concat(PREDS24,ignore_index=True)
    AUDIT24_DF.to_csv(OUT/'12_auditoria_retrospetiva_2024.csv',index=False)
    PREDS24_DF.to_csv(OUT/'13_previsoes_2024.csv',index=False)
    print('RETROSPECTIVE 2024 AUDIT (do not use for model selection)')
    display(AUDIT24_DF.round(3))

## 12. Spatial value across different tourism profiles

We assign the profiles from Notebook 05 to walk-forward predictions **for post-hoc analysis only** (the models **never received these profiles as training labels**; scenario E receives only clusters recomputed in each fold). We compare errors by profile; small cells must be interpreted cautiously. The original profiles are historical only in a retrospective sense, as they may reflect tourist attractions observed after 2024.


In [ ]:
if 'cluster' in espaco and not DELTA_DF.empty:
    map_perfis=espaco[['hotel_key','cluster']].drop_duplicates('hotel_key')
    map_perfis['cluster']=pd.to_numeric(map_perfis.cluster,errors='coerce')
    x=DELTA_DF[DELTA_DF.comparacao.isin(['B-A','D-C','E-D'])].merge(
      map_perfis,on='hotel_key',how='left',validate='many_to_one')
    PERFIL_GANHO=x.groupby(['modelo','comparacao','cluster'],dropna=True).agg(
      n=('delta_absoluto','size'),hoteis=('hotel_key','nunique'),delta_MAE=('delta_absoluto','mean')).reset_index()
    PERFIL_GANHO.to_csv(OUT/'14_ganhos_por_perfil_espacial.csv',index=False)
    fig,ax=plt.subplots(figsize=(9,4.5))
    sns.barplot(data=PERFIL_GANHO[(PERFIL_GANHO.modelo=='xgboost')&(PERFIL_GANHO.comparacao.isin(['D-C','E-D']))],
                x='cluster',y='delta_MAE',hue='comparacao',ax=ax,errorbar=None)
    ax.axhline(0,c='gray',lw=.8);ax.set(xlabel='Descriptive profile from Notebook 05',
        ylabel='ΔMAE (new − previous)',title='Value of accessibility and clusters by hotel type')
    plt.tight_layout();plt.savefig(OUT/'figura_04_heterogeneidade_perfis.png',bbox_inches='tight');plt.show()
    display(PERFIL_GANHO.query("modelo=='xgboost'").round(2))
else:
    print('Heterogeneity plot unavailable: descriptive clusters or paired predictions are missing.')

## 13. Sensitivity: extreme-value treatment and excluding 2020 from training

The main experiment retains 2020. Here, for **XGBoost in scenario D with 2023 validation**, we compare the original versus clipped training target and the exclusion of training pairs whose target year is 2020. The latter **does not completely remove information about 2020**, because 2020 productivity may still appear as `lag_y` in 2021. The 2023 evaluation sample remains fixed.


In [ ]:
SENS=[]
if EXECUTAR_SENSIBILIDADE and 'xgboost' in MODELOS:
    tr,va,mats=get_temporal_fold(2023)
    for titulo,raw,drop in [('todos_p995',False,False),('todos_original',True,False),('sem_alvo_2020_p995',False,True)]:
        try:
            if drop:
                tr2=tr.loc[tr.f_year.ne(2020)].copy()
                gr2=SpatialClusterFold(K_CLUSTERS).fit(tr2)
                prep=FeatureFold(tr2,'D',gr2)
                mat={'tr_raw':prep.transform(tr2,False),'va_raw':prep.transform(va,False),
                     'tr_scale':prep.transform(tr2,True),'va_scale':prep.transform(va,True)}
            else: tr2=tr;mat=mats['D']
            pred,extra=estimate_predictions('xgboost',tr2,va,mat,usar_clip=not raw)
            SENS.append({'politica':titulo,'n_treino':len(tr2),'n_val':len(va),
                         **metrics(va.y,pred),**extra})
        except Exception as e: print('Sensitivity analysis not run:',titulo,str(e)[:200])
SENS_DF=pd.DataFrame(SENS)
if not SENS_DF.empty:
    SENS_DF.to_csv(OUT/'15_sensibilidade_2020_outliers.csv',index=False)
    display(SENS_DF.round(3))

## 14. Interpretability: SHAP and spatial indicators (optional)

We generate **predictive**, not causal, explanations using XGBoost under scenario D in the 2023 fold. Both the model and preprocessing are fitted using earlier years only. We produce a plot of the 15 most influential features and, where descriptive profile identifiers exist, an aggregated plot of the absolute contribution of the four spatial indicators by profile.

If `shap` fails due to the installed environment version, the main benchmark results remain available and the reason is documented.


In [ ]:
SHAP_STATUS={'executado':False,'motivo':'not requested or unavailable'}
if EXECUTAR_SHAP and 'xgboost' in MODELOS:
    try:
        import shap
        tr,va,mats=get_temporal_fold(2023)
        mat=mats['D']
        model=build_model('xgboost')
        thr=float(np.quantile(tr.y,PERCENTIL_Y))
        model.fit(mat['tr_raw'],np.minimum(tr.y.to_numpy(float),thr))
        rg=np.random.default_rng(SEED)
        idx=rg.choice(len(va),size=min(250,len(va)),replace=False)
        values=shap.TreeExplainer(model).shap_values(mat['va_raw'][idx])
        values=np.asarray(values)
        nomes=mat['colnames']
        importancias=np.abs(values).mean(axis=0)
        imp_df=pd.DataFrame({'variavel':nomes,'shap_abs_medio':importancias}).sort_values('shap_abs_medio',ascending=False)
        imp_df.to_csv(OUT/'16_shap_importancias_2023.csv',index=False)
        fig,ax=plt.subplots(figsize=(9,5.2))
        part=imp_df.head(16).sort_values('shap_abs_medio')
        ax.barh(part.variavel,part.shap_abs_medio,color='#376e99')
        ax.set(xlabel='Mean |SHAP| — productivity (target units)',
               title='XGBoost-D, 2023 predictions (n=250 maximum)')
        plt.tight_layout();plt.savefig(OUT/'figura_05_shap_2023.png',bbox_inches='tight');plt.show()
        if 'cluster' in espaco:
            z=va.iloc[idx][['hotel_key']].merge(espaco[['hotel_key','cluster']],on='hotel_key',how='left',validate='many_to_one')
            feats=[i for i,c in enumerate(nomes) if c in SP_COLS]
            aux=pd.DataFrame({'perfil':z.cluster.to_numpy(),
                              'peso_espacial_SHAP':np.abs(values[:,feats]).sum(axis=1)})
            fig,ax=plt.subplots(figsize=(8,4.2))
            sns.boxplot(data=aux,x='perfil',y='peso_espacial_SHAP',showfliers=False,color='#77b5c7',ax=ax)
            ax.set(xlabel='Tourism profile (Notebook 05)',ylabel='Total spatial |SHAP|',
                   title='Predictive spatial contribution by profile, 2023')
            plt.tight_layout();plt.savefig(OUT/'figura_06_SHAP_por_perfil_2023.png',bbox_inches='tight');plt.show()
        SHAP_STATUS={'executado':True,'modelo':'xgboost','cenario':'D','ano':2023,'amostras':len(idx)}
    except Exception as e:
        SHAP_STATUS={'executado':False,'erro':type(e).__name__+': '+str(e)[:450]}
        print('SHAP unavailable (main benchmark unaffected):',SHAP_STATUS['erro'])
print('Status SHAP:',SHAP_STATUS)

## 15. Robustness test: hotels unseen during training

We reserve a subset of 2023 hotels **completely absent from model fitting** and train using earlier years from other hotels. This is more demanding than predicting future years for already-observed hotels. The reserved hotels' features at `t−1` are assumed available at prediction time, but **no target observations from these hotels are used during fitting**. Because 2023 was included in earlier analyses, this evaluation is also **retrospective**, not independent of the broader algorithm-selection process.

We use Random Forest and XGBoost only, across scenarios A–E, to investigate how well results transfer between hotels, retaining the same sample for each scenario.


In [ ]:
HOTEIS_NOVOS=[]
RNG_H=np.random.default_rng(SEED)
CANDIDATOS=PARES.loc[PARES.f_year.eq(2023),'hotel_key'].unique()
RNG_H.shuffle(CANDIDATOS)
RETIDOS=set(CANDIDATOS[:max(80,int(.25*len(CANDIDATOS)))])
TR_H=PARES.loc[(PARES.f_year<2023)&(~PARES.hotel_key.isin(RETIDOS))].copy()
TE_H=PARES.loc[PARES.f_year.eq(2023)&PARES.hotel_key.isin(RETIDOS)].copy()
assert set(TR_H.hotel_key).isdisjoint(set(TE_H.hotel_key))
assert TR_H.f_year.max()<2023 and TE_H.f_year.eq(2023).all()
MODEL_H=[x for x in ['random_forest','xgboost'] if x in MODELOS]
C_H=SpatialClusterFold(K_CLUSTERS).fit(TR_H)
for c in CENARIOS:
    prep=FeatureFold(TR_H,c,C_H)
    mat={'tr_raw':prep.transform(TR_H,False),'va_raw':prep.transform(TE_H,False),
         'tr_scale':prep.transform(TR_H,True),'va_scale':prep.transform(TE_H,True)}
    for modelo in MODEL_H:
        try:
            pred,info=estimate_predictions(modelo,TR_H,TE_H,mat)
            HOTEIS_NOVOS.append({'modelo':modelo,'cenario':c,'ano':2023,
                'hoteis_novos':TE_H.hotel_key.nunique(),'n_treino':len(TR_H),
                **metrics(TE_H.y,pred),**info})
        except Exception as e:
            print('Unseen-hotel test failed:',modelo,c,str(e)[:150])
RESULT_H=pd.DataFrame(HOTEIS_NOVOS)
if not RESULT_H.empty:
    RESULT_H.to_csv(OUT/'19_teste_hoteis_nao_observados_2023.csv',index=False)
    print('Group holdout — no test hotel appears in training:')
    display(RESULT_H.pivot(index='cenario',columns='modelo',values='MAE').round(2))

## 16. Audit, deliverables, and scientific message

This notebook **does not contain a new blind test set**, and the historical provenance of POI locations still needs verification. The objective is to **consolidate the retrospective evidence**, assess the incremental contribution of different information sources, document year-to-year differences, and prepare figures and tables for scientific review.

An MAE improvement is not automatically causal; a gain from E over D does not imply new information because the cluster is a transformation of the spatial feature vector. The ideal next independent confirmation would use data from 2025 or another hotel cohort with features and POIs available at the time predictions were made.


In [ ]:
STATUS=[]
for nome in MODELOS:
    STATUS.append({'modelo':nome,'anos_cv_concluidos':int(CV_DF[(CV_DF.modelo==nome)&CV_DF.cenario.eq('A')].ano.nunique()),
                   'cenarios_cv_concluidos':int(CV_DF[(CV_DF.modelo==nome)&CV_DF.cenario.isin(CENARIOS)].shape[0]),
                   'total_esperado':len(ANOS_CV)*len(CENARIOS),
                   'completo':int(CV_DF[(CV_DF.modelo==nome)&CV_DF.cenario.isin(CENARIOS)].shape[0])==len(ANOS_CV)*len(CENARIOS)})
STATUS_DF=pd.DataFrame(STATUS)
STATUS_DF.to_csv(OUT/'17_status_modelos_execucao.csv',index=False)
if LOG_FALHAS:
    pd.DataFrame(LOG_FALHAS).to_csv(OUT/'07_modelos_nao_executados.csv',index=False)

proveniencia={
 'projeto':'KBS Luis Silvano / Notebook 07',
 'base_anual':str(ARQ_PAINEL.resolve()),'indicadores_espaciais':str(ARQ_ESPACO.resolve()),
 'k_vizinhos':K_USADO,'indicadores':SP_COLS,'observacoes_pares':len(PARES),
 'periodo':[2018,2024],'anos_CV':ANOS_CV,'auditoria_2024_nao_cega':True,
 'perfil_execucao':PERFIL,'modelos_solicitados':MODELOS,'p995_y_treino':PERCENTIL_Y,
 'sem_2020_na_principal':False,'shap':SHAP_STATUS,'python':sys.version.split()[0],
 'pandas':pd.__version__,'sklearn':__import__('sklearn').__version__,
 'nota_temporal':'POIs may have been observed after the study period; verify provenance and date.',
 'aviso_selecao':'2021–2024 were explored in previous studies; results are retrospective, not an independent test.'}
(OUT/'18_metadata_metodologia.json').write_text(json.dumps(proveniencia,ensure_ascii=False,indent=2),encoding='utf-8')
print('\nEXECUTION STATUS BY MODEL:');display(STATUS_DF)
print('\nBEST 2021–2023 COMBINATION, for audit guidance only:',
      RANKING.iloc[0][['modelo','cenario','relativo','MAE']].to_dict())
print('Files exported to:',OUT.resolve())

## 17. Interpretation guide for writing the paper

**Results to communicate after examining the generated CSV files:**

- **Data and protocol:** seven years, independent hotels, four accessibility dimensions, and five information scenarios. Cite the collection date for each spatial source.
- **H1:** compare B−A and D−C on **the same hotel–year pairs**, highlighting MAE/RMSE and year-to-year stability. Bootstrap intervals are conditional on the observed years.
- **H2:** compare C−A and D−B; explain why a lag can help in some years and hurt in others, particularly during shock/recovery periods.
- **H3:** compare E−D and characterise the clusters from Notebook 05. If E does not outperform D, the profiles may still be valuable for communication and interpretation without adding accuracy.
- **Robustness:** compare extreme-value treatment and inclusion of 2020 in training. Do not discard the pandemic shock merely for convenience.
- **Models:** report only models that completed every fold; do not claim that modern models are inferior when their configurations are not comparable or their runs are incomplete.
- **Core limitations:** POIs potentially observed after the study period, absence of a genuinely uninspected test, possible changes in sample composition, and a short annual panel.

**Possible title:** *Spatio-Temporal Knowledge Enrichment for Hotel Labour Productivity Forecasting: Tourism Accessibility, Economic Shocks, and Interpretable Profiles.*

**Correct interpretation:** this is the final **retrospective** integrated benchmark. A genuinely new confirmatory experiment would require collecting 2025 data or an external cohort.
